In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
plt.style.use('seaborn')

excel_name = '??1 ???A??????????1.xlsx'#??1 ???A??????????.xlsx
na_values = '?'

def cp(value, value_name):
    print('==============%s=============='%value_name)
    print(value)    
    print('==============%s=============='%value_name)

data1 = pd.read_excel(excel_name, sheet_name=0,na_values=na_values)
print('??', len(data1)/72)
data1.head(3)

In [ ]:
data2 = pd.read_excel(excel_name, sheet_name=1, na_values=na_values)
print('??', len(data2)/24)
data2.head(3)

In [ ]:
data3 = pd.read_excel(excel_name, sheet_name=2, na_values=na_values)
data3.head(3)

# ????-???????,???,???

## ???????????????(????,????,??????)

In [ ]:
start_time = '2020-07-23 00:00:00'
end_time = '2021-07-13 00:00:00'
time_list = pd.date_range(start_time, end_time, freq='D')
print('????',len(time_list))
print('????', len(data1)/72)

In [ ]:
lack_list = list(set(time_list)-set(data1['??????']))
lack_list

## ???????(????????????)

In [ ]:
start_time = '2019-04-16 00:00:00'
end_time = '2021-07-13 07:00:00'
time_list = pd.date_range(start_time, end_time, freq='H')
print('????',len(time_list))
print('?????', len(time_list)-len(data2))
print('?????', 1-len(data2)/len(time_list))

In [ ]:
temp_df = data2.copy()
lack_list = list(set(time_list)-set(data2['????']))
lack_df = pd.DataFrame(lack_list,columns=['????'])
temp_df = pd.concat([temp_df,lack_df])
temp_df.sort_values("????",inplace=True)
temp_df.index = list(range(len(temp_df)))
temp_df.shape

In [ ]:
temp_df = temp_df
temp_df.iloc[:,2:] = temp_df.iloc[:,2:].interpolate()
temp_df.isnull().sum()

In [ ]:
temp_df = temp_df.applymap(lambda v: max(0,v) if isinstance(v, float) else v)
temp_df.describe()

## data1???


In [ ]:
data1.isnull().sum()

In [ ]:
data1.info()

# ????-??

## L??????????? 

In [ ]:
L=10
backward_length = 24*L
forward_length = 24*3
final_start_time = '2020-07-23 00:00:00'

for i in range(len(temp_df)):
    if temp_df['????'][i+backward_length]!=pd.Timestamp(final_start_time):
        temp_df = temp_df.drop(labels=i) 
    else:
        break
temp_df.index=range(len(temp_df))
temp_df.head(3)

In [ ]:
temp_df.tail(3)

In [ ]:
(len(temp_df)-7)/24

# ???

In [ ]:
from sklearn.preprocessing import MinMaxScaler
#minmaxscaler
mm_channel1= MinMaxScaler()
mm_channel2= MinMaxScaler()
mm_channel3= MinMaxScaler()
mm_y = MinMaxScaler()


In [ ]:
mm_y.fit_transform(temp_df.iloc[:,2:6+2])

In [ ]:
temp_df.iloc[:,2:] = mm_channel2.fit_transform(temp_df.iloc[:,2:])
temp_df

In [ ]:
data1.iloc[:,3:-6]

In [ ]:
data1.iloc[:,3:-6] = mm_channel1.fit_transform(data1.iloc[:,3:-6])
data1

In [ ]:
data1.iloc[:,-6:] = mm_channel1.fit_transform(data1.iloc[:,-6:])
data1

## ???? ??? channel2,??y

In [ ]:
x1 = []
y = []
for i in range(365-L-3+1):
    line = i*24
    x1_temp = temp_df.iloc[line:line+backward_length,2:]
    y_temp = temp_df.iloc[line+backward_length:line+backward_length+forward_length,2:2+6]
    x1.append(x1_temp)
    y.append(y_temp)
x1 = np.array(x1)
y = np.array(y)
cp(x1_temp.shape, 'x1_temp.shape')
cp(y_temp.shape, 'y_temp.shape')
cp(x1.shape, 'x1.shape')
cp(y.shape, 'y.shape')
y_temp

## ?y?72???????

In [ ]:
def squzze(matrix):
    return np.array([matrix[:24].mean(axis=0),matrix[24:48].mean(axis=0),matrix[48:72].mean(axis=0)])

y_day = np.array([squzze(inst) for inst in y])
y_day.shape

## channel1,channel3

In [ ]:
data1.shape[0]/72

In [ ]:
x2 = []
for i in range(356-3):
    x2_temp = data1.iloc[i*72:(i+1)*72,3:]
    x2.append(x2_temp)
x2 = np.array(x2)
cp(x2_temp.shape, 'x2_temp.shape')
cp(x2.shape, 'x2.shape')
x2_temp

## ?channel3?72???????

In [ ]:
channel3 = x2[:,:,-6:]
channel1 = x2[:,:,:-6]
channel3 = np.array([squzze(inst) for inst in channel3])
channel2 = x1
channel1.shape,channel2.shape,channel3.shape,y_day.shape

# ??

In [ ]:
import os
import tensorflow as tf
os.environ["CUDA_VISIBLE_DEVICES"] = '6'
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
tensorflow_version = tf.__version__
gpu_available = tf.test.is_gpu_available()
print("tensorflow_version", tensorflow_version, "GPU available", gpu_available)

In [ ]:
from tensorflow.keras.layers import Bidirectional
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Reshape, Dense, Embedding, Dropout, GRU, MaxPooling1D, Lambda, Concatenate, \
    Multiply, RepeatVector, Flatten, Activation, Permute, Conv1D
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

index_from = 1#
batch_size = 32#256
embedding_size = 100#100
rnn_units = 64
drop_rate = 0.1
num_epoch = 50
model_name_path = 'BiGRU_best_model.h5'

def modelDef():
    input_text = Input(shape=(MAX_WORDS, ))
    embeddings = Embedding(input_dim=vocab_size + index_from+1,#vocab size
                           output_dim=embedding_size,
                           mask_zero=True, input_length=MAX_WORDS)(input_text)
    bigru=Bidirectional(GRU(units=rnn_units, return_sequences=False))(embeddings)#??GRU?
    dropout = Dropout(drop_rate)(bigru)
    Softmax = Dense(label_num, activation="softmax", use_bias=True)(dropout)
    model = Model(inputs=[input_text], outputs=[Softmax])
    model.compile(optimizer="adam", loss='categorical_crossentropy',
                  metrics=['accuracy','top_k_categorical_accuracy'])
    return model

myModel = modelDef()
print(myModel.summary())
es = EarlyStopping(monitor='val_loss', patience=10)
cp = ModelCheckpoint(filepath = model_name_path, monitor='val_loss', save_best_only=True)
history = myModel.fit(x= x_train[100:100+100],
                    y= y_train[100:100+100],
                    batch_size=batch_size,
                    epochs=num_epoch,
                    verbose=1,callbacks=[es, cp],
                    validation_split=0.1)
# history = myModel.fit(x= X_train,
#                     y= y_train,
#                     batch_size=batch_size,
#                     epochs=num_epoch,
#                     verbose=2,
#                     validation_split=0.1)
# myModel.save(model_name_path)

#????
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(loss))
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.legend(loc='upper right')
plt.title('Train and Val Loss')
plt.show()


In [ ]:
import math

def compute_AQI(a):
    #a=np.array([n_SO2,n_NO2,n_PM10,n_PM25,n_O3,n_CO])
    T=np.array([[0,50,150,475,800,1600,2100,2620],[0,40,80,180,280,565,750,940],[0,50,150,250,350,420,500,600],[0,35,75,115,150,250,350,500],[0,100,160,215,265,800,0,0],[0,2,4,14,24,36,48,60]])
    I=np.array([0,50,100,150,200,300,400,500])
    BP_LO = []
    BP_Hi=[]
    IAQI_LO=[]
    IAQI_Hi=[]
    IAQI=np.zeros((1,6))
    result=[]
    for j in range(6):
        for i in range(1,9):
            if a[j] <= T[j, i]:
                BP_LO.append(T[j, i-1])
                BP_Hi.append(T[j, i])
                IAQI_LO.append(I[i-1])
                IAQI_Hi.append(I[i])
                break
            else:
                continue
        IAQI[0, j] = (IAQI_Hi[j] - IAQI_LO[j]) / (BP_Hi[j] - BP_LO[j]) * (a[j] - BP_LO[j]) + IAQI_LO[j]
        result.append(math.ceil(IAQI[0, j]))
    AQI=np.max(result)
    c=np.argmax(result)
    name=["SO2", "NO2", "PM10", "PM25", "O3", "CO"]
#     if AQI<=50:
#         print("AQI?%d" %(AQI))
#         print("????????")
#     else:
#         print("AQI?%d" % (AQI))
#         print('??????%s' % (name[c]))
    return AQI
    

AQI = [compute_AQI(inst) for inst in data2.iloc[:,2:8].values]
data2['AQI'] = AQI
fkdeplot(data2.iloc[:,-1].values, 'AQI')
